In [ ]:
import open3d as o3d
import numpy as np
from pathlib import Path
import os

# Configuration
source_folder = "/mnt/c/data/faro_scans/csic/exports/exports/"  # Parent folder containing subfolders with XYZ files
destination_folder = "/mnt/c/data/faro_scans/csic/exports/exports/combined/"  # Where to save the combined PCD files

# Create destination folder if it doesn't exist
Path(destination_folder).mkdir(parents=True, exist_ok=True)

# Iterate through each subfolder
source_path = Path(source_folder)
print("subfolders in source folder:", list(source_path.iterdir()))


#include_list = ["Pipes", "Valves"]
for subfolder in source_path.iterdir():
    if not subfolder.is_dir():
        continue
    
    # if subfolder.name not in include_list:
    #     print(f"Skipping subfolder: {subfolder.name}")
    #     continue

    print(f"Processing subfolder: {subfolder.name}")
    
    # Find all XYZ, TXT, and ASC files in this subfolder
    point_cloud_files = sorted(list(subfolder.glob("*.xyz")) + list(subfolder.glob("*.txt")) + list(subfolder.glob("*.asc")))
    
    if not point_cloud_files:
        print(f"  No XYZ, TXT, or ASC files found in {subfolder.name}")
        continue
    
    print(f"  Found {len(point_cloud_files)} point cloud files")
    
    # Load and combine all point clouds from this subfolder
    combined_pcd = None
    
    for point_file in point_cloud_files:
        #print(f"    Loading {point_file.name}")
        
        # Load point cloud file
        try:
            # Read the file, but only take the first 3 columns (x, y, z) and ignore extra scalar fields
            points = np.loadtxt(point_file, dtype=np.float32, usecols=(0, 1, 2))
            
            # Ensure points is 2D (in case there's only one point)
            if points.ndim == 1:
                points = points.reshape(1, -1)
            
            # Create Open3D point cloud
            pcd = o3d.geometry.PointCloud()
            pcd.points = o3d.utility.Vector3dVector(points)
            
            # Combine with previous point clouds
            if combined_pcd is None:
                combined_pcd = pcd
            else:
                combined_pcd.points = o3d.utility.Vector3dVector(
                    np.vstack([
                        np.asarray(combined_pcd.points),
                        np.asarray(pcd.points)
                    ])
                )
        except Exception as e:
            print(f"      Error loading {point_file.name}: {e}")
            continue
    
    # Save combined point cloud as PCD
    if combined_pcd is not None:
        output_path = Path(destination_folder) / f"{subfolder.name}.pcd"
        o3d.io.write_point_cloud(str(output_path), combined_pcd)
        print(f"  Saved combined point cloud to {output_path}")
        print(f"  Total points: {len(combined_pcd.points)}")
    else:
        print(f"  No valid point clouds to combine for {subfolder.name}")

print("\nProcessing complete!")